# 14 - Validate Packed Ternary Reconstruction

Esta etapa valida a prova prática que faltava depois do `scaled500`:

1. Ler `deploy/manifest.json` e `deploy/mlp_ternary_2bit.bin`.
2. Validar integridade estrutural dos 78 MLPs empacotados em 2-bit.
3. Reconstruir os MLPs a partir do binário packed.
4. Substituir os MLPs do `student_pruned` por esses pesos reconstruídos.
5. Aplicar o adapter `student_ternary_recovered_l78`.
6. Rodar a bateria local de 100 perguntas.

Importante: este notebook ainda **não** prova runtime eficiente direto em 2-bit. Ele prova que o pacote packed contém informação suficiente para reconstruir um modelo funcional.


In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, gc, re, time, hashlib, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
AUDIT_DIR = RESULTS_DIR / 'deployment_audit'
for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, EXPORTS_DIR, AUDIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('DEPLOY_DIR =', DEPLOY_DIR)
print('AUDIT_DIR =', AUDIT_DIR)


In [ ]:
# Dependências para reconstrução + avaliação local.
import subprocess, sys
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-U',
    'transformers',
    'accelerate',
    'sentencepiece',
    'safetensors',
    'peft',
    'datasets',
    'torchao>=0.16.0',
])


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

HF_TOKEN = os.environ.get('HF_TOKEN')
PRUNED_CHECKPOINT = CHECKPOINTS_DIR / 'student_pruned'
L78_ADAPTER = CHECKPOINTS_DIR / 'student_ternary_recovered_l78'
QUESTIONS_PATH = DATA_DIR / 'gemma4_100_questions.json'

PACKAGE_CANDIDATES = [
    DEPLOY_DIR,
    EXPORTS_DIR / 'ternary_mlp_l78',
    PROJECT_ROOT / 'deploy',
    PROJECT_ROOT / 'exports' / 'ternary_mlp_l78',
]
PACKAGE_DIR = None
for candidate in PACKAGE_CANDIDATES:
    if (candidate / 'manifest.json').exists() and (candidate / 'mlp_ternary_2bit.bin').exists():
        PACKAGE_DIR = candidate
        break
if PACKAGE_DIR is None:
    checked = [str(p) for p in PACKAGE_CANDIDATES]
    raise FileNotFoundError('Pacote packed não encontrado em: ' + json.dumps(checked, ensure_ascii=False, indent=2))

MANIFEST_PATH = PACKAGE_DIR / 'manifest.json'
BIN_PATH = PACKAGE_DIR / 'mlp_ternary_2bit.bin'

RUN_FUNCTIONAL_RECONSTRUCTION_EVAL = True
MAX_NEW_TOKENS = 24
BATCH_SIZE = 1
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'

print('PRUNED_CHECKPOINT', PRUNED_CHECKPOINT.exists(), PRUNED_CHECKPOINT)
print('L78_ADAPTER', L78_ADAPTER.exists(), L78_ADAPTER)
print('QUESTIONS_PATH', QUESTIONS_PATH.exists(), QUESTIONS_PATH)
print('PACKAGE_DIR', PACKAGE_DIR)
print('MANIFEST', MANIFEST_PATH.exists(), MANIFEST_PATH)
print('BIN', BIN_PATH.exists(), BIN_PATH, BIN_PATH.stat().st_size if BIN_PATH.exists() else None)


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def human_bytes(n):
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

manifest = load_json(MANIFEST_PATH)
questions = load_json(QUESTIONS_PATH)

assert manifest.get('format') == 'rpt_hybrid_ternary_mlp_lora_v1', manifest.get('format')
assert manifest['summary']['num_tensors'] == 78, manifest['summary']
assert BIN_PATH.exists(), BIN_PATH
assert BIN_PATH.stat().st_size == manifest['summary']['packed_total_bytes'], (BIN_PATH.stat().st_size, manifest['summary']['packed_total_bytes'])

print(json.dumps(manifest['summary'], ensure_ascii=False, indent=2))
print('bin_sha256 =', sha256_file(BIN_PATH))


In [ ]:
# Funções 2-bit: codebook 0=zero, 1=+scale, 2=-scale, 3=reserved.
def unpack_codes_from_blob(blob, n):
    raw = np.frombuffer(blob, dtype=np.uint8)
    codes = np.empty(raw.size * 4, dtype=np.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    return codes[:n]


def unpack_tensor_from_blob(blob, n, shape, scale, dtype=torch.float32):
    codes = unpack_codes_from_blob(blob, n)
    vals = np.zeros(n, dtype=np.float32)
    vals[codes == 1] = scale
    vals[codes == 2] = -scale
    tensor = torch.from_numpy(vals).view(tuple(shape))
    return tensor.to(dtype=dtype)


def tensor_code_hist_from_blob(blob, n):
    codes = unpack_codes_from_blob(blob, n)
    counts = np.bincount(codes, minlength=4).astype(np.int64)
    return {str(i): int(counts[i]) for i in range(4)}

# Validação estrutural completa, sem carregar modelo.
tensors = manifest['tensors']
structural_errors = []
structural_rows = []
expected_offset = 0
code_totals = np.zeros(4, dtype=np.int64)
selected_indexes = sorted(set([0, 1, 2, len(tensors)//2, len(tensors)-3, len(tensors)-2, len(tensors)-1]))
selected_reconstruction_checks = []

with open(BIN_PATH, 'rb') as f:
    for idx, item in enumerate(tensors):
        if item['offset'] != expected_offset:
            structural_errors.append({'name': item['name'], 'error': 'non_contiguous_offset', 'expected': expected_offset, 'actual': item['offset']})
        expected_offset = item['offset'] + item['packed_bytes']

        expected_packed = (item['numel'] + 3) // 4
        if item['packed_bytes'] != expected_packed:
            structural_errors.append({'name': item['name'], 'error': 'packed_size_mismatch', 'expected': expected_packed, 'actual': item['packed_bytes']})

        f.seek(item['offset'])
        blob = f.read(item['packed_bytes'])
        if len(blob) != item['packed_bytes']:
            structural_errors.append({'name': item['name'], 'error': 'short_read', 'expected': item['packed_bytes'], 'actual': len(blob)})
            continue

        hist = tensor_code_hist_from_blob(blob, item['numel'])
        counts = np.array([hist[str(i)] for i in range(4)], dtype=np.int64)
        code_totals += counts
        reserved_ratio = counts[3] / max(1, item['numel'])
        if counts[3] != 0:
            structural_errors.append({'name': item['name'], 'error': 'reserved_code_present', 'reserved_count': int(counts[3])})

        structural_rows.append({
            'name': item['name'],
            'shape': item['shape'],
            'numel': item['numel'],
            'scale': item['scale'],
            'packed_bytes': item['packed_bytes'],
            'fp16_bytes': item['fp16_bytes'],
            'compression_vs_fp16': item['compression_vs_fp16'],
            'code_histogram': hist,
            'zero_ratio': round(counts[0] / item['numel'], 8),
            'positive_ratio': round(counts[1] / item['numel'], 8),
            'negative_ratio': round(counts[2] / item['numel'], 8),
            'reserved_ratio': round(reserved_ratio, 8),
        })

        if idx in selected_indexes:
            recon = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'])
            uniq = sorted(float(x) for x in torch.unique(recon).tolist())
            selected_reconstruction_checks.append({
                'name': item['name'],
                'shape': item['shape'],
                'unique_values_count': len(uniq),
                'unique_values': uniq,
                'abs_max': float(recon.abs().max().item()),
                'mean_abs': float(recon.abs().mean().item()),
            })
            del recon

structural_summary = {
    'manifest_path': str(MANIFEST_PATH),
    'bin_path': str(BIN_PATH),
    'bin_size_bytes': BIN_PATH.stat().st_size,
    'bin_size_human': human_bytes(BIN_PATH.stat().st_size),
    'bin_sha256': sha256_file(BIN_PATH),
    'num_tensors': len(tensors),
    'total_numel': int(sum(x['numel'] for x in tensors)),
    'expected_final_offset': expected_offset,
    'offset_matches_file_size': expected_offset == BIN_PATH.stat().st_size,
    'manifest_summary': manifest['summary'],
    'code_totals': {str(i): int(code_totals[i]) for i in range(4)},
    'global_zero_ratio': round(float(code_totals[0]) / max(1, int(code_totals.sum())), 8),
    'global_positive_ratio': round(float(code_totals[1]) / max(1, int(code_totals.sum())), 8),
    'global_negative_ratio': round(float(code_totals[2]) / max(1, int(code_totals.sum())), 8),
    'global_reserved_ratio': round(float(code_totals[3]) / max(1, int(code_totals.sum())), 8),
    'structural_errors': structural_errors,
    'selected_reconstruction_checks': selected_reconstruction_checks,
}

structural_report = {
    'status': 'passed' if not structural_errors and structural_summary['offset_matches_file_size'] else 'failed',
    'summary': structural_summary,
    'tensor_rows': structural_rows,
}
structural_report_path = AUDIT_DIR / 'packed_ternary_reconstruction_structural.json'
save_json(structural_report, structural_report_path)
print('saved', structural_report_path)
print(json.dumps(structural_summary, ensure_ascii=False, indent=2)[:8000])
if structural_report['status'] != 'passed':
    raise RuntimeError('Structural packed validation failed; inspect structural_errors.')


In [ ]:
# Avaliação local: mesma lógica dos notebooks anteriores.
def normalize_text(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())


def judge_answer(expected, answer):
    e = normalize_text(expected)
    a = normalize_text(answer)
    if not e:
        return False
    return e in a or a in e


def extract_answer(prompt, generated):
    text = generated
    if prompt in text:
        text = text.split(prompt, 1)[-1]
    text = text.strip()
    text = text.split('\n')[0].strip()
    return text


def run_eval(model, tokenizer, questions, batch_size=1, max_new_tokens=24, verbose=True):
    rows = []
    correct = 0
    device = next(model.parameters()).device
    for idx, q in enumerate(questions):
        prompt = PROMPT_TEMPLATE.format(question=q['question'])
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = extract_answer(prompt, generated)
        ok = judge_answer(q['expected_answer'], answer)
        correct += int(ok)
        row = {
            'id': q.get('id', idx + 1),
            'category': q.get('category'),
            'question': q['question'],
            'expected_answer': q['expected_answer'],
            'model_answer': answer,
            'is_correct': ok,
            'raw_output': generated,
        }
        rows.append(row)
        if verbose:
            print(f"{idx+1:03d}/{len(questions)}", 'OK' if ok else 'ERR', '|', q['question'], '=>', answer)
    by_category = {}
    for row in rows:
        cat = row.get('category') or 'unknown'
        by_category.setdefault(cat, {'total': 0, 'correct': 0})
        by_category[cat]['total'] += 1
        by_category[cat]['correct'] += int(row['is_correct'])
    for cat, vals in by_category.items():
        vals['accuracy'] = vals['correct'] / vals['total'] if vals['total'] else 0.0
    summary = {
        'total_questions': len(questions),
        'correct': correct,
        'accuracy': correct / len(questions) if questions else 0.0,
        'by_category': by_category,
    }
    return rows, summary


In [ ]:
# Reconstrói MLPs packed dentro do student_pruned e aplica o adapter l78.
# Esta é a prova funcional do pacote packed reconstruído.
functional_result = None
if RUN_FUNCTIONAL_RECONSTRUCTION_EVAL:
    cleanup_memory()
    if not PRUNED_CHECKPOINT.exists():
        raise FileNotFoundError(f'PRUNED_CHECKPOINT ausente: {PRUNED_CHECKPOINT}')
    if not L78_ADAPTER.exists():
        raise FileNotFoundError(f'L78_ADAPTER ausente: {L78_ADAPTER}')

    tokenizer = AutoTokenizer.from_pretrained(str(PRUNED_CHECKPOINT), token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        str(PRUNED_CHECKPOINT),
        dtype='auto',
        device_map='auto',
        token=HF_TOKEN,
    )
    model.eval()

    named_params = dict(model.named_parameters())
    missing_params = []
    meta_params = []
    applied = []
    start = time.time()

    with open(BIN_PATH, 'rb') as f:
        for item in manifest['tensors']:
            name = item['name']
            param = named_params.get(name)
            if param is None:
                missing_params.append(name)
                continue
            if getattr(param, 'is_meta', False):
                meta_params.append(name)
                continue
            f.seek(item['offset'])
            blob = f.read(item['packed_bytes'])
            recon = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'], dtype=torch.float32)
            if tuple(param.shape) != tuple(recon.shape):
                raise RuntimeError(f'Shape mismatch for {name}: param={tuple(param.shape)} recon={tuple(recon.shape)}')
            with torch.no_grad():
                param.copy_(recon.to(device=param.device, dtype=param.dtype))
            applied.append(name)
            del recon

    if missing_params or meta_params:
        raise RuntimeError(json.dumps({'missing_params': missing_params, 'meta_params': meta_params}, ensure_ascii=False, indent=2))
    if len(applied) != 78:
        raise RuntimeError(f'Expected 78 packed MLP tensors applied, got {len(applied)}')

    adapter_model = PeftModel.from_pretrained(model, str(L78_ADAPTER), token=HF_TOKEN, is_trainable=False)
    adapter_model.eval()

    rows, eval_summary = run_eval(adapter_model, tokenizer, questions, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS, verbose=True)
    elapsed = round(time.time() - start, 2)

    functional_result = {
        'status': 'completed',
        'source_pruned_checkpoint': str(PRUNED_CHECKPOINT),
        'adapter': str(L78_ADAPTER),
        'packed_manifest': str(MANIFEST_PATH),
        'packed_bin': str(BIN_PATH),
        'applied_packed_mlp_tensors': len(applied),
        'elapsed_seconds_after_model_load_start': elapsed,
        'summary': eval_summary,
        'rows': rows,
        'note': 'This evaluates reconstructed packed MLP tensors injected into student_pruned plus the l78 PEFT adapter. It still materializes tensors into torch weights; it is not an efficient 2-bit runtime kernel.',
    }
    functional_path = AUDIT_DIR / 'packed_ternary_reconstruction_functional_eval.json'
    save_json(functional_result, functional_path)
    print('saved', functional_path)
    print(json.dumps(eval_summary, ensure_ascii=False, indent=2))

    del adapter_model, model
    cleanup_memory()
else:
    print('RUN_FUNCTIONAL_RECONSTRUCTION_EVAL=False; pulando avaliação funcional.')


In [ ]:
# Relatório final consolidado e zip de artefatos.
final_report = {
    'status': 'completed' if structural_report['status'] == 'passed' and (functional_result is None or functional_result.get('status') == 'completed') else 'incomplete',
    'what_this_proves': 'The packed 2-bit MLP artifact can be structurally decoded and, when materialized back into torch tensors with the l78 adapter, can be evaluated as a functional reconstructed model.',
    'what_this_does_not_prove': 'It does not prove efficient inference directly over the packed 2-bit representation. The next step is runtime/memory benchmarking and custom module/kernel work.',
    'structural_report_path': str(AUDIT_DIR / 'packed_ternary_reconstruction_structural.json'),
    'functional_report_path': str(AUDIT_DIR / 'packed_ternary_reconstruction_functional_eval.json') if functional_result else None,
    'structural_summary': structural_summary,
    'functional_summary': functional_result.get('summary') if functional_result else None,
    'expected_reference_accuracy': {
        'local_l78_previous': 0.92,
        'external_scaled500_delta_range': '[-0.006, +0.006] vs student_pruned_base',
    },
}
final_report_path = AUDIT_DIR / 'packed_ternary_reconstruction_report.json'
save_json(final_report, final_report_path)
print('saved', final_report_path)
print(json.dumps(final_report, ensure_ascii=False, indent=2)[:10000])

artifact_paths = [
    AUDIT_DIR / 'packed_ternary_reconstruction_structural.json',
    AUDIT_DIR / 'packed_ternary_reconstruction_report.json',
]
if functional_result:
    artifact_paths.append(AUDIT_DIR / 'packed_ternary_reconstruction_functional_eval.json')

zip_path = AUDIT_DIR / 'packed_ternary_reconstruction_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in artifact_paths:
        if p.exists():
            zf.write(p, arcname=p.name)
print('ZIP salvo em:', zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
